# Tokenizer Evaluation — 4 variants × 6 vocab sizes

Compares **four tokenizer variants** trained via `scripts/train_all.py`:

1. **SentencePiece Unigram** (Kudo et al. 2018 — deterministic Viterbi encode)
2. **Unigram + Subword Regularization** (same model; sampled segmentations at encode time)
3. **SentencePiece WordPiece** (BERT-style merges)
4. **Byte-level BPE** (GPT-2 / RoBERTa style)

Vocab sizes: **500, 1K, 5K, 10K, 20K, 30K**.

Metrics: fertility, **Compression Factor (CF)**, encode/decode timing, round-trip fidelity, segmentation plots.

Run `python scripts/evaluate_tokenizers.py` for a full JSON export, or execute this notebook interactively.

## 1. Setup

In [ ]:
import json
import sys
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "tokenizer_utils.py").exists():
    ROOT = ROOT / "Tokenization"
sys.path.insert(0, str(ROOT))

from tokenizer_utils import (
    TOKENIZER_LABELS,
    TOKENIZER_TYPES,
    VOCAB_SIZES,
    classify_script,
    compression_factor,
    discover_available_models,
    fertility_for_words,
    load_heldout_docs,
    load_tokenizer,
    time_encode_decode,
)

plt.rcParams["font.family"] = ["Segoe UI", "Tahoma", "Arial", "DejaVu Sans"]
try:
    import arabic_reshaper
    from bidi.algorithm import get_display

    def display_ar(text):
        return get_display(arabic_reshaper.reshape(text))
except ImportError:
    def display_ar(text):
        return text

heldout_docs = load_heldout_docs(ROOT / "data" / "heldout_docs.jsonl")
texts = [d["text"] for d in heldout_docs]
WORDS = [(w, classify_script(w)) for t in texts for w in t.split() if w.strip()]

available = discover_available_models()
print(f"held-out docs: {len(heldout_docs):,}")
print(f"trained (key, vocab) pairs on disk: {len(available)}")
if not available:
    raise FileNotFoundError("no models found — run: python scripts/train_all.py")

DEFAULT_VOCAB = 20_000 if any(v == 20_000 for _, v in available) else available[-1][1]
print(f"default comparison vocab: {DEFAULT_VOCAB:,}")

## 2. Full evaluation sweep

Runs CF, fertility, timing, and round-trip checks for every trained `(tokenizer, vocab_size)` pair.

In [ ]:
rows = []
for key, vocab_size in available:
    tok = load_tokenizer(key, vocab_size)
    timing = time_encode_decode(tok, texts, rounds=2)
    cf_vals = [compression_factor(t, tok.pieces) for t in texts]
    fert = fertility_for_words(WORDS, tok.pieces)
    mismatches = sum(1 for t in texts if tok.decode(tok.encode(t)) != t)
    rows.append({
        "tokenizer_key": key,
        "tokenizer": TOKENIZER_LABELS[key],
        "vocab_size": vocab_size,
        "cf_mean": sum(cf_vals) / len(cf_vals),
        "fertility": fert["overall_fertility"],
        "encode_ms": timing["encode_ms_per_doc"],
        "decode_ms": timing["decode_ms_per_doc"],
        "roundtrip_ms": timing["roundtrip_ms_per_doc"],
        "roundtrip_mismatches": mismatches,
    })

eval_df = pd.DataFrame(rows).sort_values(["vocab_size", "tokenizer_key"])
print(eval_df.to_string(index=False))

out_path = ROOT / "data" / "eval_results.json"
out_path.write_text(
    json.dumps({"heldout_docs": len(heldout_docs), "results": rows}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"\nsaved -> {out_path}")

## 3. Compression Factor (CF) by vocab size

$$\text{CF} = \frac{\text{total effective tokens}}{\text{total characters} + \text{total words}}$$

For a word containing `<unk>`: effective cost = `len(word) + 1`.

**Lower CF → better compression** (fewer splits / less UNK inflation).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for key in TOKENIZER_TYPES:
    sub = eval_df[eval_df["tokenizer_key"] == key]
    if sub.empty:
        continue
    ax.plot(sub["vocab_size"], sub["cf_mean"], marker="o", label=TOKENIZER_LABELS[key])
ax.set_xscale("log")
ax.set_xlabel("vocab size")
ax.set_ylabel("Compression Factor (mean, held-out)")
ax.set_title("CF vs vocab size — lower is better")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Encode / decode timing

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
metrics = [("encode_ms", "encode ms/doc"), ("decode_ms", "decode ms/doc"), ("roundtrip_ms", "round-trip ms/doc")]
for ax, (col, title) in zip(axes, metrics):
    for key in TOKENIZER_TYPES:
        sub = eval_df[eval_df["tokenizer_key"] == key]
        if sub.empty:
            continue
        ax.plot(sub["vocab_size"], sub[col], marker="o", label=TOKENIZER_LABELS[key])
    ax.set_xscale("log")
    ax.set_xlabel("vocab size")
    ax.set_ylabel(title)
    ax.grid(True, alpha=0.3)
axes[0].legend(fontsize=7, loc="upper left")
plt.suptitle("Tokenizer speed on held-out set (2 timing rounds)")
plt.tight_layout()
plt.show()

## 5. Fertility at default vocab (by script)

In [ ]:
fertility_rows = []
for key in TOKENIZER_TYPES:
    if (key, DEFAULT_VOCAB) not in set(available):
        continue
    tok = load_tokenizer(key, DEFAULT_VOCAB)
    fert = fertility_for_words(WORDS, tok.pieces)
    for r in fert["rows"]:
        fertility_rows.append({"tokenizer": TOKENIZER_LABELS[key], **r})

fertility_df = pd.DataFrame(fertility_rows)
print(fertility_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4.5))
pivot = fertility_df[fertility_df["script"] != "other"].pivot(
    index="script", columns="tokenizer", values="fertility"
)
pivot = pivot.reindex([s for s in ["arabic", "latin", "mixed", "ALL"] if s in pivot.index])
pivot.plot.bar(ax=ax)
ax.set_ylabel("tokens per word")
ax.set_title(f"Fertility by script @ vocab={DEFAULT_VOCAB:,}")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

## 6. Round-trip fidelity

In [ ]:
rt = eval_df[["tokenizer", "vocab_size", "roundtrip_mismatches"]].copy()
rt["heldout_docs"] = len(heldout_docs)
print(rt.to_string(index=False))

## 7. Segmentation visualizations

Side-by-side token boxes for all four tokenizers at `DEFAULT_VOCAB`. Boxes are laid out LTR in token order (known simplification for RTL text).

In [ ]:
PALETTE = ["#a6cee3", "#b2df8a", "#fdbf6f", "#fb9a99", "#cab2d6", "#ffff99", "#8dd3c7", "#fccde5"]


def segments_for(tok, text):
    if tok.key == "bpe":
        from tokenizer_utils import bpe_model_path
        enc = __import__("tokenizers").Tokenizer.from_file(str(bpe_model_path(tok.vocab_size)))
        out = []
        for id_, raw in zip(enc.encode(text).ids, enc.encode(text).tokens):
            label = enc.decode([id_]).strip() or raw
            out.append(display_ar(label) or "·")
        return out
    return [display_ar(p.replace("\u2581", " ")).strip() or "·" for p in tok.pieces(text)]


def plot_segments(ax, segments, title):
    ax.set_title(title, loc="left", fontsize=9)
    ax.set_xlim(0, max(len(segments), 1))
    ax.set_ylim(0, 1)
    ax.axis("off")
    for i, seg in enumerate(segments):
        ax.add_patch(plt.Rectangle((i, 0), 1, 1, facecolor=PALETTE[i % len(PALETTE)], edgecolor="white"))
        ax.text(i + 0.5, 0.5, seg, ha="center", va="center", fontsize=10)


def compare_all_tokenizers(text, label):
    active = [k for k in TOKENIZER_TYPES if (k, DEFAULT_VOCAB) in set(available)]
    fig, axes = plt.subplots(len(active), 1, figsize=(max(6, 0.8 * len(text)), 1.8 * len(active)))
    if len(active) == 1:
        axes = [axes]
    for ax, key in zip(axes, active):
        tok = load_tokenizer(key, DEFAULT_VOCAB)
        segs = segments_for(tok, text)
        plot_segments(ax, segs, f"{label} — {TOKENIZER_LABELS[key]}")
    plt.tight_layout()
    plt.show()

In [ ]:
EXAMPLES = [
    ("راني عارف", "Darija Arabic"),
    ("bezzaf", "Arabizi"),
    ("wach kayen chkoun hna", "Darija Latin"),
    ("راني نروح للخدمة demain", "code-switched"),
]
for sentence, label in EXAMPLES:
    compare_all_tokenizers(sentence, label)

## 8. CF heatmap (tokenizer × vocab size)

In [ ]:
pivot_cf = eval_df.pivot(index="tokenizer", columns="vocab_size", values="cf_mean")
pivot_cf = pivot_cf.reindex([TOKENIZER_LABELS[k] for k in TOKENIZER_TYPES if k in eval_df["tokenizer_key"].values])

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(pivot_cf.values, aspect="auto", cmap="YlOrRd_r")
ax.set_xticks(range(len(pivot_cf.columns)))
ax.set_xticklabels([f"{c:,}" for c in pivot_cf.columns], rotation=45)
ax.set_yticks(range(len(pivot_cf.index)))
ax.set_yticklabels(pivot_cf.index, fontsize=8)
ax.set_title("Compression Factor heatmap (lower / greener is better)")
plt.colorbar(im, ax=ax, label="CF mean")
plt.tight_layout()
plt.show()